# EDA I: datos, objetivo y relaciones entre variables

Acompaña las sub-pestañas **Datos y calidad, Variable objetivo, Univariado, Bivariado, Multivariado y Fuga de datos** del dashboard. Cada figura es la misma del dashboard; después de cada una va su interpretación y, en los recuadros *Detalle*, lo que el dashboard no muestra: las fórmulas de las pruebas y las tablas completas. Todas las decisiones usan las semanas 2010–2021.

In [1]:
import os  # Rutas
import sys  # Configuración del intérprete
sys.path.insert(0, os.path.abspath(".."))  # Carpeta entregable2 (paquete tablero)
sys.path.insert(0, os.path.abspath("../.."))  # Raíz del proyecto (paquete src)
sys.path.insert(0, os.path.abspath("../../entregable1"))  # Paquete e1 (código fuente que se muestra)
import warnings  # Control de avisos
warnings.filterwarnings("ignore")  # Oculta avisos de versiones
import numpy as np  # Cálculo numérico
import pandas as pd  # Tablas
from IPython.display import HTML, Markdown, display  # Incrustar figuras y texto
from tablero import figuras as fg  # Las mismas figuras del dashboard
from tablero.datos import A, M, MOD, FEATURES, curvas, metricas_umbral, rezago_espacial  # Los mismos artefactos del dashboard
pd.set_option("display.width", 160)  # Ancho de las tablas
pd.set_option("display.max_columns", 30)  # Columnas visibles
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")  # Cuatro decimales


def mostrar(fig):  # Muestra una figura del dashboard como HTML interactivo
    """Pone el fondo oscuro de las tarjetas del dashboard y carga Plotly desde internet."""
    fig.update_layout(paper_bgcolor=fg.SUPERFICIE, plot_bgcolor=fg.SUPERFICIE, margin=dict(l=40, r=20, t=50, b=40))  # Fondo oscuro
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False, config={"displaylogo": False})))  # Figura interactiva
cal = A["calidad"]  # Calidad de los datos
print(f"Temperatura imputada: {cal['frac_imputada']:.1%} de las semanas")  # Faltante oculto
cal["patron"].to_frame("municipios")  # Patrón por municipio

Temperatura imputada: 64.3% de las semanas


,municipios
temp_mean_missing,
siempre imputada,226
parcialmente,129
nunca imputada,21


## Datos y calidad

### Patrón de los faltantes

In [2]:
mostrar(fg.fig_matriz_faltantes(cal["matriz_faltantes"]))  # Matriz municipio × año
mostrar(fg.fig_temp_imputada(cal["hist_temp"]))  # Observada frente a imputada

**Interpretación.** No hay celdas nulas, pero la temperatura está imputada en el 64 % de las semanas. El patrón es por **municipio** (226 nunca medidos, 21 siempre medidos, 129 parcialmente) y estable en el tiempo. La imputación original es casi una constante cerca de 22,4 °C: no conserva la diferencia climática entre municipios.

### Mecanismo de los faltantes

:::{admonition} Detalle: prueba de Little
Con $J$ patrones de faltantes, $\boldsymbol\mu$ y $\boldsymbol\Sigma$ estimadas por máxima verosimilitud (algoritmo EM) y $\bar{\mathbf x}_j$ la media de las variables observadas en el patrón $j$:

$$d^2 = \sum_{j=1}^{J} n_j\,(\bar{\mathbf x}_j - \boldsymbol\mu_j)^\top \boldsymbol\Sigma_j^{-1} (\bar{\mathbf x}_j - \boldsymbol\mu_j) \sim \chi^2\Big(\sum_j p_j - p\Big)\ \text{bajo MCAR}.$$

La implementación (EM incluido) está en `entregable1/e1/estadistica.py`.
:::

In [3]:
li = cal["little"]  # Prueba de Little
print(f"Little: d² = {li['d2']:,.1f}, gl = {li['gl']}, p = {li['p']:.1e}")  # Resultado
pd.Series(cal["smd"], name="diferencia estandarizada (faltante − observado)").to_frame()  # Tamaños de efecto

Little: d² = 2,304.8, gl = 5, p = 0.0e+00


,diferencia estandarizada (faltante − observado)
prec_total,-0.0149
ndvi_mean,0.2808
oni_anom,0.0080
log_poblacion,-0.6801
log_casos,-0.3347


**Interpretación.** Little rechaza MCAR. La diferencia más grande es el tamaño del municipio ($d \approx -0{,}7$): faltan las mediciones en municipios pequeños, sin estación meteorológica. Es **MAR estructural**; un componente MNAR no es verificable. Decisión: conservar la bandera `temp_missing` (método del indicador) en lugar de reimputar con vecinos.

### Valores imposibles, atípicos y representatividad

In [4]:
pd.Series(cal["chequeos"], name="filas con problema").to_frame()  # Consistencia

,filas con problema
Filas duplicadas exactas,0
"Claves (municipio, semana) repetidas",0
Suma de edades ≠ total,0
Hombres + mujeres ≠ total,0
"Temperatura fuera de [0, 40] °C",0
"NDVI fuera de [−1, 1]",0
Conteos o lluvia negativos,0
Semanas que no avanzan 7 días,0


In [5]:
display(cal["atipicos"].to_frame())  # Atípicos por la regla de Tukey
cal["top_casos"]  # Las semanas con más casos

,atípicos (%)
casos_totales,11.7997
temp_mean,18.1375
prec_total,4.6682
ndvi_mean,1.6621
oni_anom,2.3018
Flujo_in,12.5646
poblacion,14.0957


,municipio,departamento,week_start,casos_totales
0,Santiago De Cali,Valle Del Cauca,2024-05-20,1426
1,Santiago De Cali,Valle Del Cauca,2024-06-03,1399
2,Santiago De Cali,Valle Del Cauca,2024-05-27,1378
3,Santiago De Cali,Valle Del Cauca,2024-06-17,1366
4,Santiago De Cali,Valle Del Cauca,2024-06-10,1364
5,Santiago De Cali,Valle Del Cauca,2024-04-01,1284
6,Santiago De Cali,Valle Del Cauca,2024-04-29,1281
7,Santiago De Cali,Valle Del Cauca,2024-04-15,1277


In [6]:
rep = cal["representatividad"]  # Cobertura por región
print(f"Población cubierta en 2024: {cal['pob_cubierta'] / 1e6:.1f} millones")  # Cobertura poblacional
rep

Población cubierta en 2024: 32.2 millones


,país,panel,cobertura,participación panel,participación país
region,,,,,
Andina,629,173,0.2750,0.4601,0.5606
Caribe,197,99,0.5025,0.2633,0.1756
Pacífica,179,44,0.2458,0.1170,0.1595
Orinoquía,59,41,0.6949,0.1090,0.0526
Amazonía,58,19,0.3276,0.0505,0.0517


**Interpretación.** No hay valores imposibles ni duplicados. Los atípicos de casos son epidemias reales (Cali en 2024) y se conservan con transformación logarítmica; el 18 % de "atípicos" de temperatura es un artefacto de la imputación. El panel cubre un tercio de los municipios pero ≈ 60 % de la población, con el Caribe y la Orinoquía sobrerrepresentados.

## Variable objetivo

In [7]:
o = A["objetivo"]  # Objetivo
mostrar(fg.fig_prevalencia_anual(o["anual"]))  # Prevalencia por año
mostrar(fg.fig_mes_region(o["mes"], o["region"]))  # Por mes y región
o["anual"].rename(columns={"mean": "prevalencia", "sum": "brotes", "size": "filas"})

,prevalencia,brotes,filas
ANO,,,
2015,0.1281,2119,16544
2016,0.0906,1772,19552
2017,0.0167,326,19552
2018,0.0724,1416,19552
2019,0.2532,4951,19552
2020,0.0992,1977,19928
2021,0.0600,1083,18048


**Interpretación.** La prevalencia del brote va del 1,7 % (2017) al 25 % (2019): el desbalance depende del año. La exactitud no sirve para decidir (no alertar nunca acierta el 90 %); se usan AUC-PR, MCC y Brier, y la validación recorre años completos. La estacionalidad mensual del objetivo es débil porque el canal endémico ya la descuenta.

In [8]:
mostrar(fg.fig_objetivo_tiempo(o["nacional"]))  # Municipios en brote
mapa = M.merge(o["mapa"].reset_index(), on="COD_MUN_N")  # Prevalencia por municipio
mostrar(fg.fig_mapa_valor(mapa, "prevalencia", "brote", ".0%", (0, 0.3)))  # Mapa
mapa["prevalencia"].describe().to_frame().T

,count,mean,std,min,25%,50%,75%,max
prevalencia,376.0000,0.1028,0.0848,0.0000,0.0340,0.0822,0.1671,0.4108


**Interpretación.** Los brotes llegan en ondas nacionales (2015–2016 y 2019). En el espacio la prevalencia varía entre municipios (mediana ≈ 8 %), pero no sigue el nivel de casos, porque cada municipio se compara consigo mismo.

## Análisis unidimensional

In [9]:
u = A["univariado"]  # Univariado
for v in ["casos_totales", "temp_mean", "prec_total", "ndvi_mean"]:  # Variables representativas
    mostrar(fg.fig_histograma(u["hist"][v], v))  # Histograma + caja
u["tabla"]  # Estadísticos, forma, atípicos y normalidad

,media,mediana,desv. est.,p5,p95,asimetría,curtosis,atípicos IQR (%),p normalidad (K²)
variable,,,,,,,,,
casos_totales,3.5549,0.0000,20.2744,0.0000,12.0000,18.2714,473.3343,10.0979,0.0000
temp_mean,23.1829,22.4633,3.0148,18.2039,28.6500,-0.2163,2.1400,18.1569,0.0000
prec_total,42.5322,25.1000,51.8553,0.0000,145.0000,2.2587,8.3742,4.6377,0.0000
ndvi_mean,0.6307,0.6502,0.1147,0.4119,0.7852,-0.7837,0.4180,1.6132,0.0000
oni_anom,0.0238,-0.1400,0.8715,-1.2000,1.9100,0.8874,1.1323,4.3131,0.0000
oni_total,27.0263,27.1300,0.9927,25.4300,28.8300,0.1406,-0.4647,0.0000,0.0000
Flujo_in,2.9785,0.5896,16.4615,0.0841,9.5308,20.5873,576.3910,12.5374,0.0000
poblacion,"85,757.9309","22,490.0000","410,379.4074","6,233.0000","299,173.0000",14.3056,233.9893,14.0957,0.0000


**Interpretación.** Los casos tienen asimetría > 18 y curtosis en cientos (mediana 0): se usa $\log(1+x)$. La temperatura es bimodal por la imputación; la lluvia es asimétrica a la derecha; el NDVI, levemente a la izquierda. Ninguna variable es normal (todas las pruebas rechazan aun con 5 000 filas), lo que justifica pruebas no paramétricas.

In [10]:
pd.DataFrame({"departamentos raros (< 3 municipios)": [", ".join(u["categoricas"]["departamento"][u["categoricas"]["departamento"] < 3].index)],
              "semanas por fase ENSO": [u["categoricas"]["enso"].to_dict()], "municipios por región": [u["categoricas"]["region"].to_dict()]}).T

,0
departamentos raros (< 3 municipios),"Guainía, Quindío, Vaupés, Bogotá, D.C., Guavia..."
semanas por fase ENSO,"{'Neutral': 287, 'La Niña': 179, 'El Niño': 160}"
municipios por región,"{'Amazonía': 19, 'Andina': 173, 'Caribe': 99, ..."


**Interpretación.** El departamento tiene categorías raras y no se codifica (haría memorizar lugares); la semana 53 (2015 y 2020) se resuelve con seno y coseno; ninguna fase ENSO es rara.

## Análisis bidimensional

:::{admonition} Detalle: medidas de asociación usadas
* **Correlación biserial de rangos** (Mann-Whitney): $r_{rb} = 2U/(n_1 n_0) - 1 \in [-1, 1]$; es $2\,\text{AUC} - 1$.
* **Kruskal-Wallis** con tamaño de efecto $\varepsilon^2 = H/(n-1)$ (0,01 pequeño, 0,08 mediano, 0,26 grande).
* **V de Cramér** con corrección de sesgo: $V = \sqrt{\tilde\varphi^2/\min(\tilde k - 1, \tilde r - 1)}$.
* **Información mutua** $I(X;Y) = \sum p(x,y)\ln\frac{p(x,y)}{p(x)p(y)}$, estimada por vecinos más cercanos; capta relaciones no lineales.
* **VIF** $= 1/(1-R_j^2) = [\mathbf R^{-1}]_{jj}$, con $\mathbf R$ la matriz de correlación.
* **Holm** controla la tasa de error de la familia; **Benjamini-Hochberg**, la tasa de falsos descubrimientos.
:::

In [11]:
b = A["bivariado"]  # Bivariado
for p in b["pares"]:  # Pares numéricos
    x, y = p.split("|")  # Variables
    mostrar(fg.fig_densidad(b["pares"][p], x, y))  # Densidad conjunta
pd.DataFrame({p: {"Pearson": d["pearson"], "Spearman": d["spearman"], "n": d["n"]} for p, d in b["pares"].items()}).T

,Pearson,Spearman,n
temp_mean_m4|prec_total_m4,0.0555,-0.0942,"48,070.0000"
lc_m8|lc_l0,0.9105,0.7803,"132,728.0000"
temp_mean_m8|lc_m8,0.0509,0.0353,"48,070.0000"
oni_l8|lc_m8,0.1576,0.1884,"132,728.0000"


**Interpretación.** Temperatura y lluvia están poco relacionadas; la historia de casos es muy persistente (Pearson 0,91 frente a Spearman 0,78: la correlación lineal la dominan las semanas epidémicas); la temperatura casi no se correlaciona con los casos; el ONI rezagado sí, de forma positiva.

In [12]:
mostrar(fg.fig_correlacion(b["corr"]))  # Spearman
b["vif"].sort_values(ascending=False).to_frame("VIF").T  # Multicolinealidad

,temp_mean_m8,temp_mean_m12,temp_mean_m4,lc_m8,temp_mean_m1,lc_m12,ndvi_mean_m8,ndvi_mean_m4,prec_total_m8,ndvi_mean_m12,prec_total_m12,ndvi_mean_m1,prec_total_m4,lc_l3,lc_l2,...,flujo_m4,lpob,lc_l0,brote_l0,prec_total_m1,l_umbral_obj,incid,prec_anom_m8,ndvi_anom_m8,lc_m4,temp_missing,temp_anom_m8,ratio_umbral,sem_sin,sem_cos
VIF,"1,026.9446",525.3422,357.4618,122.0889,79.4624,61.5032,55.6818,29.8368,27.3910,24.2848,15.2052,9.5105,8.7865,6.5461,6.3211,...,4.5551,4.2437,4.2049,3.4583,2.4502,2.2941,1.9784,1.5736,1.5219,1.4684,1.3404,1.2660,1.2402,1.1576,1.1439


**Interpretación.** Las medias móviles de una misma variable forman bloques con $|\rho| > 0{,}9$ y VIF ≫ 10 (hasta ≈ 1 000 en la temperatura de 8 semanas). La predicción no sufre, pero los coeficientes individuales son inestables: se usa penalización L2.

In [13]:
mostrar(fg.fig_asociacion(b["asociacion"], "Información mutua"))  # Asociación
b["asociacion"].sort_values("mi", ascending=False)  # Tabla completa (no aparece en el dashboard)

,r_rb,p,auc,mi,p_holm,q_fdr
variable,,,,,,
brote_m8,0.7323,0.0000,0.8662,0.1192,0.0000,0.0000
incid,0.6932,0.0000,0.8466,0.1076,0.0000,0.0000
ratio_umbral,0.6123,0.0000,0.8061,0.1072,0.0000,0.0000
lc_m4,0.7625,0.0000,0.8812,0.1040,0.0000,0.0000
lc_m8,0.7539,0.0000,0.8769,0.1023,0.0000,0.0000
lc_m12,0.7376,0.0000,0.8688,0.0933,0.0000,0.0000
lc_l0,0.7044,0.0000,0.8522,0.0924,0.0000,0.0000
brote_l0,0.5402,0.0000,0.7701,0.0911,0.0000,0.0000
lc_l1,0.6828,0.0000,0.8414,0.0880,0.0000,0.0000


**Interpretación.** La historia de casos domina (AUC univariados de 0,77–0,88). El flujo, el ONI y la población tienen asociación moderada. El clima local tiene efectos pequeños aunque casi todo es significativo tras Holm y FDR: con $n$ grande el tamaño de efecto manda.

In [14]:
for v in ["temp_mean_m8", "prec_total_m8", "oni_l8", "lc_m4"]:  # Relaciones no lineales
    mostrar(fg.fig_deciles(b["deciles"][v], v))  # Tasa de brote por decil

**Interpretación.** Con la temperatura medida, el riesgo sube hacia los 25–28 °C y se estabiliza; con la historia de casos crece de forma monótona y fuerte. Las formas no lineales anticipan una limitación de los modelos base lineales.

In [15]:
display(b["kruskal"])  # Por región
display(b["cramer"])  # Categóricas
b["enso"]  # Brote por fase ENSO

,H,p,ε²,p (Holm)
variable,,,,
lc_l0,835.9909,0.0000,0.0063,0.0000
temp_mean_m8,"19,859.6053",0.0000,0.4131,0.0000
prec_total_m8,"15,390.8373",0.0000,0.1160,0.0000
ndvi_mean_m8,"6,353.1595",0.0000,0.0479,0.0000


,categorías,χ²,gl,p,V de Cramér,p (Holm)
variable,,,,,,
MES,12,85.3226,11,0.0000,0.0237,0.0000
region,5,152.5075,4,0.0000,0.0334,0.0000
departamento,32,876.4571,31,0.0000,0.0798,0.0000
fase_enso,3,"3,700.7494",2,0.0000,0.1669,0.0000
temp_missing,2,359.0001,1,0.0000,0.0519,0.0000


y,0,1
fase_enso,,
La Niña,0.9548,0.0452
Neutral,0.9169,0.0831
El Niño,0.8318,0.1682


**Interpretación.** La temperatura difiere muchísimo entre regiones ($\varepsilon^2 \approx 0{,}41$) y la historia de casos casi nada. La fase ENSO es la categórica más asociada al brote (V ≈ 0,17): 16,8 % en El Niño frente a 4,5 % en La Niña.

## Análisis multivariado

In [16]:
mu = A["multivariado"]  # Multivariado
mostrar(fg.fig_pca(mu["varianza"], mu["pca_xy"], mu["pca_y"]))  # PCA
acum = np.cumsum(mu["varianza"])  # Varianza acumulada
print(f"Componentes para 80 %: {np.searchsorted(acum, 0.8) + 1} · para 90 %: {np.searchsorted(acum, 0.9) + 1} · autovalor > 1: {(mu['autovalores'] > 1).sum()}")
mu["cargas"].round(3)  # Cargas completas de las 4 primeras componentes

Componentes para 80 %: 8 · para 90 %: 12 · autovalor > 1: 7


,CP1,CP2,CP3,CP4
lc_l0,0.3060,-0.0260,0.0690,-0.0160
lc_l1,0.3020,-0.0230,0.0620,-0.0080
lc_l2,0.3020,-0.0220,0.0620,-0.0070
lc_l3,0.3010,-0.0210,0.0600,-0.0060
lc_m4,0.3230,-0.0240,0.0670,-0.0100
lc_m8,0.3210,-0.0210,0.0620,-0.0060
lc_m12,0.3170,-0.0180,0.0570,-0.0030
brote_l0,0.2020,-0.0330,0.0780,-0.0630
brote_m8,0.2370,-0.0320,0.0780,-0.0620
ratio_umbral,0.0700,-0.0040,0.0540,-0.0950


**Interpretación.** 8 componentes explican el 80 % de la varianza: la dimensionalidad efectiva es mucho menor que 34. CP1 es la historia de casos; CP2–CP3, temperatura y NDVI; CP4, lluvia. Los brotes no se separan linealmente en dos dimensiones.

In [17]:
mostrar(fg.fig_tsne(mu["tsne"], mu["tsne_y"], mu["tsne_region"], "region"))  # t-SNE por región
display(mu["atipicos"])  # Atípicos multivariados
mu["atipicos_municipios"].to_frame("filas atípicas")

,atípicos (%),tasa de brote en atípicos,tasa de brote en el resto
"Mahalanobis (χ² 99,9 %)",6.3107,0.3147,0.0885
Isolation Forest,14.7135,0.3886,0.0535
Ambos métodos,4.7013,0.3885,0.0887


,filas atípicas
municipio,
"Bogotá, D.C.",306
Santiago De Cali,287
Guamal,223
Florencia,205
Medellín,187
Buenaventura,186
Valledupar,143
Villavicencio,136


**Interpretación.** t-SNE agrupa más por región que por objetivo: el brote es un estado transitorio, no una subpoblación. Los atípicos multivariados tienen una tasa de brote cuatro veces mayor (semanas epidémicas, grandes ciudades): se conservan.

:::{admonition} Detalle: distancia de Mahalanobis
$D^2(\mathbf x) = (\mathbf x - \boldsymbol\mu)^\top \boldsymbol\Sigma^{-1}(\mathbf x - \boldsymbol\mu) \sim \chi^2_{34}$ bajo normalidad; se marca atípico por encima del percentil 99,9. Por la colinealidad se usa la pseudo-inversa de $\boldsymbol\Sigma$.
:::

In [18]:
mostrar(fg.fig_grupos(mu["perfil"]))  # Grupos de municipios
print("Silueta por k:", {k: round(v, 3) for k, v in mu["silueta"].items()})  # Elección de k
mu["perfil"].groupby("grupo")[["prevalencia", "temp", "lluvia", "lpob", "flujo"]].mean()  # Perfil de cada grupo

Silueta por k: {2: 0.327, 3: 0.221, 4: 0.225, 5: 0.226, 6: 0.197, 7: 0.196, 8: 0.188}


,prevalencia,temp,lluvia,lpob,flujo
grupo,,,,,
0,0.0816,23.3201,43.1404,9.7559,0.4192
1,0.1924,22.9541,35.8226,12.0193,1.7437


**Interpretación.** La mejor silueta es 0,33 con $k = 2$: estructura débil a moderada. Los grupos se separan por tamaño y movilidad, no por clima; los municipios grandes y conectados tienen más del doble de semanas en brote.

## Fuga de datos

In [19]:
f = A["fuga"]  # Auditoría
mostrar(fg.fig_auc_univariado(f["auc_univariado"]))  # AUC univariado
f["demo"].to_frame("AUC-ROC de validación cruzada")  # Demostración

,AUC-ROC de validación cruzada
Correcto: años hacia adelante con purga,0.8985
Fuga temporal: semanas mezcladas al azar,0.9064
Fuga directa: casos de la semana objetivo,0.9941


**Interpretación.** Ninguna predictora supera 0,9 de AUC univariado; los casos de la semana objetivo (información del futuro) llegan a ≈ 0,96 y se descartan. Con la validación correcta el AUC es ≈ 0,90; mezclando semanas sube un poco, y con la fuga directa llega a ≈ 0,99. El umbral objetivo $u_{t+4}$ usa semanas $t-258$ a $t-46$, todas anteriores a $t$: no es fuga.